# Chemistry landscape analysis

We’ll analyse a reaction-condition landscape using experimental yields, from selecting the design variables to exploring local optima, ruggedness and interactions. Electrochemical reduction of benzaldehyde will serve as our example.

## 1. Importing necessary dependencies

Install GraphFLA with pip, then import the dependencies below.

In [ ]:
!pip install graphfla

In [1]:
import graphfla
from graphfla.landscape import Landscape
import pandas as pd

## 2. Loading the experimental dataset

For our example, we use the benzaldehyde electroreduction experiments from [Eisenberg et al. (2025)](https://doi.org/10.1021/jacs.5c10757). The screen combines four electrode materials, three pH values and three applied potentials, giving 36 reaction conditions. Benzaldehyde can form benzyl alcohol or couple to form the dimer hydrobenzoin.

We’ll maximize the **dimer yield**. The curated CSV calculates this as conversion × dimer selectivity / 100, using the measurements in the paper’s supporting table.

| Column | Meaning |
|---|---|
| `electrode` | Electrode material: Bi, Cu, Pb or graphite. |
| `pH` | Tested pH: 2, 7 or 13. |
| `potential_V` | Applied potential in volts relative to the reversible hydrogen electrode (RHE). |
| `conversion_pct` | Percentage of benzaldehyde converted. |
| `sel_dimerization_pct` | Percentage selectivity toward dimerization. |
| `dimer_yield` | Calculated dimer yield, expressed as a percentage. |

Let’s load the measurements and inspect a few conditions.

In [2]:
data_url = (
    "https://raw.githubusercontent.com/COLA-Laboratory/GraphFLA/"
    "main/data/Chemistry/EChem/aldehyde_electro_36.csv"
)
df = pd.read_csv(data_url)[[
    "electrode", "pH", "potential_V",
    "conversion_pct", "sel_dimerization_pct", "dimer_yield",
]]
df.head()

,electrode,pH,potential_V,conversion_pct,sel_dimerization_pct,dimer_yield
0,Bi,2,-1.0,52.55,76.88,40.4004
1,Bi,2,-0.8,66.68,80.17,53.4574
2,Bi,2,-0.6,37.03,76.39,28.2872
3,Cu,2,-1.0,16.06,9.49,1.5241
4,Cu,2,-0.8,19.04,18.26,3.4767


## 3. Preparing the inputs

To construct a landscape, GraphFLA only needs two inputs—much like the features `X` and targets `y` used to train a machine-learning model:

- `X`: the experimental configurations, as a DataFrame or a two-dimensional NumPy array.
- `f`: one measured outcome per configuration, as a Series, list or one-dimensional array.

The rows of `X` and `f` must refer to the same configurations in the same order.


We select electrode, pH and potential as the controllable variables in `X`, and dimer yield as `f`. Conversion and selectivity are measured outcomes used to calculate the yield, so they stay out of `X`.

In [3]:
X = df[['electrode', 'pH', 'potential_V']]
f = df['dimer_yield']
X.head()

,electrode,pH,potential_V
0,Bi,2,-1.0
1,Bi,2,-0.8
2,Bi,2,-0.6
3,Cu,2,-1.0
4,Cu,2,-0.8


## 4. Constructing the landscape

We can combine different variable types in one `Landscape`: electrode is categorical, while pH and potential are ordinal.

This defines a neighbouring condition as changing the electrode, or moving one level along pH or potential, with the other settings fixed. An ordinal step means an adjacent **tested level**: pH 2 ↔ 7 ↔ 13, and −1.0 ↔ −0.8 ↔ −0.6 V. It does not interpolate unmeasured conditions.

We set `maximize=True` because higher dimer yield is better.

In [4]:
landscape = Landscape(maximize=True)
landscape.build_from_data(
    X, f,
    data_types={
        "electrode": "categorical",
        "pH": "ordinal",
        "potential_V": "ordinal",
    },
)

Building Landscape from data...


 - Validating data types dictionary...


   - Data types dictionary validation successful.


 - Handling missing values and duplicates...


 - Checking for missing values...


...

   - Found 3 local optima.


 - Determining global optimum...


   - Global optimum found at index 25 with fitness 76.2200.


Landscape built successfully.



Landscape(kind='default', maximize=True)

## 5. Inspecting the landscape

Let’s first look at what we’ve built. Printing the landscape shows its variables, configurations, improving edges and local optima:

In [5]:
print(landscape)

Landscape(kind='default'): 3 variables, 36 configurations, 102 edges, 3 local optima


For a closer look at individual reaction conditions, we can call `get_data()`. The selected outcome appears as `fitness`; `out_degree` counts directly improving moves, `in_degree` counts incoming moves from lower-scoring neighbours, and `is_lo` indicates local-optimum membership.

In [6]:
landscape.get_data()

,electrode,pH,potential_V,fitness,in_degree,out_degree,is_lo
0,Bi,2,-1.0,40.4004,4,1,False
1,Bi,2,-0.8,53.4574,6,0,True
2,Bi,2,-0.6,28.2872,2,3,False
3,Cu,2,-1.0,1.5241,1,4,False
4,Cu,2,-0.8,3.4767,2,4,False
5,Cu,2,-0.6,13.2771,2,3,False
6,Pb,2,-1.0,21.0553,3,2,False
7,Pb,2,-0.8,42.8239,5,1,False
8,Pb,2,-0.6,42.1465,3,2,False
9,graphite,2,-1.0,20.5028,2,3,False


We can also summarise the dimer yields with pandas:

In [7]:
landscape.get_data()["fitness"].describe()

count    36.000000
mean     28.948014
std      19.885090
min       0.198000
25%      11.990375
50%      28.125600
75%      42.315850
max      76.220000
Name: fitness, dtype: float64

To inspect the condition with the highest dimer yield, we can use the global-optimum node index:

In [8]:
landscape[landscape.go_index]

{'electrode': 'Bi',
 'pH': np.int64(13),
 'potential_V': np.float64(-0.8),
 'fitness': np.float64(76.22),
 'in_degree': 6,
 'out_degree': 0,
 'is_lo': True}

## 6. Analysing the landscape

Next, we’ll count local optima, measure ruggedness, examine interactions and assess the trend toward the best observed configuration.

In [9]:
from graphfla import analysis

### 6.1 Number of local optima

We can start with `landscape.n_lo`. A local optimum has no improving route out of its immediate neighbourhood, including neutral moves within a plateau. Each connected neutral optimum plateau counts once. Multiple local optima mean that successive improvements can lead to different endpoints.

In [10]:
print(f"Number of local optima: {landscape.n_lo}")

Number of local optima: 3


### 6.2 Fitness autocorrelation

To explore ruggedness, we can measure how similar the scores remain along random walks through neighbouring configurations. Positive autocorrelation indicates persistence of similar scores across nearby steps; lower values indicate less persistence under the same walk settings.

We use 200 walks of 20 visited configurations and a fixed seed. The walks follow stored edges in either direction and exclude separately stored neutral pairs.

In [11]:
walk_autocorrelation = analysis.autocorrelation(
    landscape,
    walk_length=20,
    walk_times=200,
    lag=1,
    seed=42,
)
print(f"Lag-1 fitness autocorrelation: {walk_autocorrelation:.4f}")

Lag-1 fitness autocorrelation: 0.3102


### 6.3 Interactions between reaction conditions

The effect of changing pH may depend on the electrode or potential. We can examine these response interactions with `classify_epistasis()`. Here, the biological term *epistasis* describes context dependence between experimental settings.

We can explore these patterns with `classify_epistasis()`, which classifies four-configuration graph motifs:

| Category | Interpretation |
|---|---|
| `magnitude` | Changes retain their direction of effect across contexts. |
| `sign` | One change reverses its direction of effect across contexts. |
| `reciprocal_sign` | Both changes reverse their direction of effect across contexts. |

The output gives their proportions among the detected motifs. The `positive` and `negative` fields form a separate partition.

In [12]:
interactions = analysis.classify_epistasis(landscape, seed=42)
interactions

EpistasisClassification(magnitude=0.6363636363636364, sign=0.2727272727272727, reciprocal_sign=0.09090909090909091, positive=0.7840909090909091, negative=0.21590909090909094)

### 6.4 Fitness-distance correlation

Finally, we can use `fdc` to check whether higher-scoring configurations tend to lie closer to the best observed configuration—a simple view of navigability.


Distance combines the number of electrode choices that differ with the number of tested pH and potential levels separating two conditions. Each step follows the variable types chosen during construction.

We use Spearman correlation to compare ranks. A negative value means higher scores tend to occur nearer the optimum; a value near zero indicates little monotonic association. This trend does not guarantee an improving path from every configuration.

In [13]:
fitness_distance_r = analysis.fdc(landscape, method="spearman")
print(f"Fitness-distance correlation: {fitness_distance_r:.4f}")

 - Calculating distances to global optimum using mixed_distance...


   - Distances to GO calculated and added as node attribute 'dist_go'.


Fitness-distance correlation: -0.4760


## 7. Running several analyses together

Once we’ve tried the metrics individually, we can collect them with `analysis.profile()`. Here, we combine the ruggedness measure and fitness-distance correlation in a pandas Series. Setting `on_error="raise"` displays errors immediately.

In [14]:
analysis.profile(
    landscape,
    include=["autocorrelation", "fdc"],
    params={"autocorrelation": {"walk_times": 200}},
    seed=42,
    on_error="raise",
    progress=False,
)

autocorrelation    0.310169
fdc               -0.476028
dtype: float64

## 8. Analysis reference

For further exploration, we can choose from the **30 analysis functions** below. Each takes a built `landscape` as its first argument. In these functions, *fitness* refers to the outcome selected as `f`.

Walsh–Hadamard and higher-order regression are included for reference. Their cost grows rapidly with the number of variables and levels, so we leave them out of the worked examples.

| Group | Function | Result |
| --- | --- | --- |
| Fitness | `fitness_distribution` | Dictionary of distribution summaries, including skewness, kurtosis, CV and Cauchy location. |
| Fitness | `fitness_effect_distribution` | List of signed effects for one mutation across matched backgrounds. |
| Ruggedness | `local_optima_ratio` | Local-optimum count divided by configuration count. |
| Ruggedness | `r_s_ratio` | Additive-model residual roughness divided by mean absolute coefficient. |
| Ruggedness | `autocorrelation` | Fitness autocorrelation along sampled walks. |
| Ruggedness | `gradient_intensity` | Mean absolute change on improving edges divided by mean fitness. |
| Correlation | `fdc` | Fitness-distance correlation: Spearman or Pearson. |
| Correlation | `neighbor_fitness_correlation` | Correlation between fitness and mean neighbour fitness: Pearson, Spearman or Kendall. |
| Correlation | `basin_fitness_correlation` | Correlation between peak fitness and greedy basin size: Spearman or Pearson. |
| Correlation | `fitness_flattening_index` | Trend in successive fitness gains along greedy walks: Spearman or Pearson. |
| Navigability | `global_optima_accessibility` | Fraction of configurations with an improving route to the global optimum. |
| Navigability | `local_optima_accessibility` | DataFrame of accessibility values for specified local optima. |
| Navigability | `mean_distance_to_global_optimum` | Mean sequence/configuration distance to the global optimum. |
| Navigability | `mean_distance_to_local_optima` | DataFrame of mean sequence/configuration distances to specified local optima. |
| Navigability | `mean_path_length_to_global_optimum` | Mean shortest graph-path length, excluding unreachable starts. |
| Navigability | `mean_path_length_to_local_optima` | DataFrame of shortest-path summaries, excluding unreachable starts. `lo=None` selects the global optimum. |
| Robustness | `neutrality` | Fraction of neighbour pairs within the fitness-difference threshold. |
| Robustness | `evolvability_enhancing_mutations` | Fraction of improving edges that also increase mean neighbour fitness by more than epsilon. |
| Robustness | `single_mutation_effects` | DataFrame of mutation-pair effects and one-sided tests at one position; positive or negative tests. |
| Robustness | `all_mutation_effects` | DataFrame of mutation-pair effects and one-sided tests at all positions. |
| Epistasis | `idiosyncratic_index` | Normalized spread of one mutation’s effects across backgrounds. |
| Epistasis | `global_idiosyncratic_index` | Mean idiosyncratic index across eligible mutations. |
| Epistasis | `gamma` | Correlation of mutation effects across matched two-site backgrounds. |
| Epistasis | `gamma_star` | Consistency of mutation-effect signs across backgrounds. |
| Epistasis | `diminishing_returns_index` | Association of background fitness with mean beneficial gain: Pearson, Spearman or regression slope. |
| Epistasis | `increasing_costs_index` | Association of background fitness with mean deleterious cost: Pearson, Spearman or regression slope. |
| Epistasis | `higher_order_epistasis` | In-sample R² of a model including interactions up to the specified order. |
| Epistasis | `walsh_hadamard` | DataFrame of additive/interaction coefficients; the reference depends on the input type. |
| Epistasis | `classify_epistasis` | Object with magnitude, sign, reciprocal-sign, positive and negative epistasis proportions. |
| Epistasis | `extradimensional_bypass` | Object with bypass proportion, mean length and motif counts. |

### Data source

Eisenberg J. B., Lee K., Schmidt J. R. and Choi K.-S. (2025). [*Understanding the Competition between Alcohol Formation and Dimerization during Electrochemical Reduction of Aromatic Carbonyl Compounds*. Journal of the American Chemical Society 147, 41390–41403](https://doi.org/10.1021/jacs.5c10757).

The repository CSV is curated from Supporting Information Table S1. Dimer yield is derived from the reported conversion and selectivity.